# Linux Kernel — Epicentra zmian: Propagation Analysis

## Teza postera
> *"Zmiana mniej niż 0.01 % funkcji jądra Linuxa może teoretycznie dotknąć ponad połowy systemu."*

### Plan analizy
| Sekcja | Pytanie | Miara |
|--------|---------|-------|
| H1 | Jak daleko rozchodzi się zmiana? | Forward Reachability / Propagation Cost |
| H2 | Jak *szybko* rozchodzi się zmiana? | BFS depth layers |
| H3 | Które subsystemy są strukturalnie kruche? | Blast-radius heatmapa |
| H4 | Kto to jest *prawdziwe* epicentrum? | Fragility Score (composite) |

### Kluczowe pojęcia
- **Stożek wpływu** funkcji `v` = zbiór wszystkich funkcji osiągalnych z `v` w grafie DAG wywołań
- **Propagation Cost (PC)** = `|stożek| / N` — frakcja systemu dotknięta zmianą
- **Fragility Score** = PC × betweenness × (1 / in_community_density) — im wyższy, tym groźniejszy


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from collections import defaultdict, deque
from scipy import stats
import time

DATA = Path("../data/out")
FIG  = Path("../figures/poster")
FIG.mkdir(parents=True, exist_ok=True)

# Styl posterowy
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300,
    "font.size": 13, "axes.titlesize": 16, "axes.labelsize": 14,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.5, "patch.linewidth": 1.4,
})
WONG = ["#0072B2","#E69F00","#009E73","#CC79A7",
        "#F0E442","#56B4E9","#D55E00","#999999"]

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    print(f"  saved → {name}.png")

# ── Wczytaj dane ────────────────────────────────────────────────────────────
print("Wczytuję dane…")
metrics   = pd.read_csv(DATA / "node_metrics.csv")
edges_raw = pd.read_csv(DATA / "edges.csv")
N = len(metrics)
print(f"Węzłów: {N:,}    Krawędzi: {len(edges_raw):,}")

# String-ID → int-indeks (poprawka z notebooków 06/07)
id_to_idx = {fid: i for i, fid in enumerate(metrics["id"].values)}

edges_idx = edges_raw.copy()
edges_idx["src_idx"] = edges_idx["source"].map(id_to_idx)
edges_idx["tgt_idx"] = edges_idx["target"].map(id_to_idx)
edges_idx = edges_idx.dropna(subset=["src_idx","tgt_idx"])
edges_idx["src_idx"] = edges_idx["src_idx"].astype(int)
edges_idx["tgt_idx"] = edges_idx["tgt_idx"].astype(int)
print(f"Krawędzi po mapowaniu: {len(edges_idx):,}")

# Tablice sąsiedztwa (listy) — tylko out-adj potrzebne do forward-BFS
out_adj = defaultdict(list)
for s, t in zip(edges_idx["src_idx"].values, edges_idx["tgt_idx"].values):
    out_adj[s].append(t)
print("Listy sąsiedztwa gotowe.")


## H1. Forward Reachability — stożek wpływu każdej funkcji

Dla każdego węzła `v` robimy BFS *w przód* i liczymy ile węzłów jest osiągalnych.

**Propagation Cost** PC(v) = reach(v) / N

Aby nie liczyć N BFS-ów na 466k węzłach, stosujemy **próbkowanie**:
losujemy K węzłów i interpolujemy rozkład. K = 20 000 daje precyzję < 1 %.


In [ ]:
K_SAMPLE = 20_000          # liczba węzłów do próbkowania
MAX_DEPTH = 999            # pełny BFS (bez ograniczenia głębokości)
rng = np.random.default_rng(42)
sample_nodes = rng.choice(N, size=min(K_SAMPLE, N), replace=False)

def forward_reach(src, adj, max_depth=MAX_DEPTH):
    """Zwraca (count, depth_of_first_driver) — BFS w przód."""
    visited = {src}
    frontier = [src]
    depth = 0
    while frontier and depth < max_depth:
        next_f = []
        for u in frontier:
            for v in adj[u]:
                if v not in visited:
                    visited.add(v)
                    next_f.append(v)
        frontier = next_f
        depth += 1
    return len(visited) - 1   # minus sam siebie

print(f"Liczę forward-reach dla {K_SAMPLE:,} próbkowanych węzłów…")
t0 = time.time()
reach_counts = np.zeros(len(sample_nodes), dtype=np.int32)
for i, v in enumerate(sample_nodes):
    reach_counts[i] = forward_reach(int(v), out_adj)
    if (i+1) % 2000 == 0:
        pct = 100*(i+1)/len(sample_nodes)
        elapsed = time.time()-t0
        eta = elapsed/(i+1)*(len(sample_nodes)-i-1)
        print(f"  {pct:5.1f}%  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

pc = reach_counts / N   # Propagation Cost ∈ [0,1]
print(f"\nGotowe za {time.time()-t0:.0f}s")
print(f"  median PC = {np.median(pc):.4f}  ({np.median(pc)*100:.2f}%)")
print(f"  mean   PC = {np.mean(pc):.4f}")
print(f"  max    PC = {pc.max():.4f}  ({pc.max()*100:.1f}%)  ← największy stożek")
print(f"  węzłów z PC>0.5 : {(pc>0.5).sum():,}  ({100*(pc>0.5).mean():.2f}%)")
print(f"  węzłów z PC>0.1 : {(pc>0.1).sum():,}  ({100*(pc>0.1).mean():.2f}%)")

# Zapisz wyniki z powrotem do metrics (tylko dla próbki)
pc_series = pd.Series(np.nan, index=range(N))
pc_series.iloc[sample_nodes] = pc
metrics["propagation_cost"] = pc_series.values


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel A: histogram PC ---
ax = axes[0]
bins = np.linspace(0, pc.max()+0.01, 60)
ax.hist(pc, bins=bins, color=WONG[0], edgecolor="white", linewidth=0.4)

# Annotacje kluczowych progów
for thresh, label, col in [(0.01,"1%",WONG[1]), (0.10,"10%",WONG[6]), (0.50,"50%","red")]:
    n = (pc >= thresh).sum()
    if n > 0:
        ax.axvline(thresh, color=col, lw=2, linestyle="--")
        ax.text(thresh+0.005, ax.get_ylim()[1]*0.85,
                f"{n:,} funkcji\n≥{label*1} PC",
                color=col, fontsize=11, fontweight="bold")

ax.set_xlabel("Propagation Cost PC(v)  [frakcja systemu dotknięta zmianą]")
ax.set_ylabel("Liczba funkcji")
ax.set_title("(A) Rozkład Propagation Cost", fontweight="bold")
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3, which="both")

# Inset: CCDF
axin = ax.inset_axes([0.55, 0.25, 0.42, 0.40])
sorted_pc = np.sort(pc)
ccdf_y = 1 - np.arange(1, len(sorted_pc)+1)/len(sorted_pc)
axin.loglog(sorted_pc[sorted_pc>0]+1e-6, ccdf_y[sorted_pc>0],
            color=WONG[0], lw=2)
axin.set_xlabel("PC", fontsize=9)
axin.set_ylabel("P(X≥pc)", fontsize=9)
axin.set_title("CCDF", fontsize=9)
axin.grid(alpha=0.3, which="both")

# --- Panel B: PC vs in-degree scatter ---
ax = axes[1]
has_pc = ~np.isnan(metrics["propagation_cost"])
x = metrics.loc[has_pc, "in_degree"].values + 1
y = metrics.loc[has_pc, "propagation_cost"].values
# Subsample 8k dla prędkości
rng2 = np.random.default_rng(7)
idx_plot = rng2.choice(len(x), size=min(8000, len(x)), replace=False)
sc = ax.scatter(x[idx_plot], y[idx_plot],
                s=6, c=np.log1p(x[idx_plot]),
                cmap="rocket_r", alpha=0.5, edgecolors="none", rasterized=True)
ax.set_xscale("log")
ax.set_xlabel("in-degree + 1  (log)")
ax.set_ylabel("Propagation Cost PC(v)")
ax.set_title("(B) Czy popularne funkcje mają duży stożek wpływu?", fontweight="bold")
plt.colorbar(sc, ax=ax, label="log(in-degree+1)")
ax.grid(alpha=0.3)

# Korelacja Spearmana
rho, p = stats.spearmanr(x, y)
ax.text(0.97, 0.97, f"Spearman ρ = {rho:.3f}\np = {p:.1e}",
        transform=ax.transAxes, ha="right", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=1.5))

fig.suptitle("H1. Propagation Cost — jak daleko rozchodzi się zmiana?",
             fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "08_h1_propagation_cost")
plt.show()


## H2. Szybkość propagacji — ile kroków do zalania systemu?

Dla top epicentrów (najwyższy PC) robimy BFS warstwami i patrzymy jak stożek *rośnie*
w funkcji odległości.

Analogia dla odbiorcy: *"jak szybko COVID rozchodzi się w sieci kontaktów"* —
ale tutaj węzłem jest funkcja a krawędzią wywołanie.


In [ ]:
def bfs_layers(src, adj, max_depth=20):
    """Zwraca listę: liczba nowych węzłów na każdej głębokości 1,2,..."""
    visited = {src}
    frontier = [src]
    layers = []
    for _ in range(max_depth):
        nxt = []
        for u in frontier:
            for v in adj[u]:
                if v not in visited:
                    visited.add(v)
                    nxt.append(v)
        if not nxt:
            break
        layers.append(len(nxt))
        frontier = nxt
    return layers

# Wybierz top 8 epicentrów (najwyższy PC) + 3 typowe (median PC)
top_mask = ~np.isnan(metrics["propagation_cost"])
top_df = metrics[top_mask].nlargest(8, "propagation_cost")
med_pc  = np.nanmedian(metrics["propagation_cost"])
# 3 węzły zbliżone do mediany
median_df = metrics[top_mask].iloc[
    (metrics.loc[top_mask, "propagation_cost"] - med_pc).abs().argsort()[:3]
]

print("Top 8 epicentrów (najwyższy PC):")
print(top_df[["func_name","subsystem","in_degree","out_degree",
              "propagation_cost"]].to_string(index=False))

# Policz warstwy BFS
layer_data = {}
all_nodes = list(top_df.index) + list(median_df.index)
labels_nodes = (["epic_"+str(i) for i in range(len(top_df))] +
                ["med_"+str(i)  for i in range(len(median_df))])

for label, node_idx in zip(labels_nodes, all_nodes):
    src = int(metrics.loc[node_idx, "id"] if "id" not in metrics.columns
              else node_idx)
    # metrics.index == positional index == nk index
    src_nk = node_idx
    layers = bfs_layers(src_nk, out_adj, max_depth=20)
    layer_data[label] = layers
    fname = metrics.loc[node_idx, "func_name"]
    pc_val = metrics.loc[node_idx, "propagation_cost"]
    print(f"  {fname[:30]:<30s}  PC={pc_val:.3f}  layers={len(layers)}  "
          f"total_reach={sum(layers)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel A: kumulatywny wzrost stożka (procent systemu) ---
ax = axes[0]
max_depth_plot = max(len(v) for v in layer_data.values())

for i, label in enumerate(labels_nodes):
    layers = layer_data[label]
    cumsum = np.cumsum(layers) / N * 100
    depths = np.arange(1, len(cumsum)+1)
    is_epic = label.startswith("epic")
    node_idx = all_nodes[i]
    fname = metrics.loc[node_idx, "func_name"][:20]
    if is_epic and i < 3:   # etykieta tylko dla top 3
        ax.plot(depths, cumsum, lw=3, color=WONG[i % 4],
                label=f"{fname} (PC={metrics.loc[node_idx,'propagation_cost']:.2f})",
                zorder=5)
    elif is_epic:
        ax.plot(depths, cumsum, lw=1.5, color=WONG[0], alpha=0.3)
    else:
        ax.plot(depths, cumsum, lw=2, color=WONG[7], linestyle="--", alpha=0.7)

# Linia mediany
med_layers_arr = np.zeros(max_depth_plot)
med_labels = [l for l in labels_nodes if l.startswith("med_")]
for label in med_labels:
    arr = np.array(layer_data[label])
    padded = np.pad(arr, (0, max_depth_plot - len(arr)))
    med_layers_arr += padded / len(med_labels)
med_cumsum = np.cumsum(med_layers_arr) / N * 100
ax.plot(range(1, max_depth_plot+1), med_cumsum, lw=3, color=WONG[7],
        linestyle="--", label="typowa funkcja (mediana PC)", zorder=6)

ax.axhline(50, color="red", lw=1.5, linestyle=":", alpha=0.7)
ax.text(0.5, 51, "50% systemu", color="red", fontsize=10)
ax.set_xlabel("Głębokość propagacji (liczba kroków wywołań)")
ax.set_ylabel("% systemu w stożku wpływu")
ax.set_title("(A) Jak szybko rośnie stożek wpływu?", fontweight="bold")
ax.legend(fontsize=9, loc="lower right", framealpha=0.95)
ax.grid(alpha=0.3)
ax.set_xlim(0, max_depth_plot + 1)
ax.set_ylim(-2, 105)

# --- Panel B: rozkład "głębokości 50%" ---
# Dla każdego próbkowanego węzła: ile kroków do osiągnięcia 50% systemu?
depths_to_half = []
for v in sample_nodes:
    layers = bfs_layers(int(v), out_adj, max_depth=25)
    cumsum = np.cumsum(layers) / N
    idx_50 = np.searchsorted(cumsum, 0.5)
    if idx_50 < len(layers):
        depths_to_half.append(idx_50 + 1)

ax = axes[1]
if len(depths_to_half) > 0:
    counts_d = np.bincount(depths_to_half)
    xs = np.arange(len(counts_d))
    ax.bar(xs[xs>0], counts_d[xs>0], color=WONG[2], edgecolor="black")
    ax.set_xlabel("Kroki potrzebne do osiągnięcia 50% systemu")
    ax.set_ylabel("Liczba funkcji (z tych które to osiągają)")
    ax.set_title(f"(B) {len(depths_to_half):,} funkcji może zalać >50% systemu\n"
                 f"Ile kroków zajmuje im dotarcie do połowy?", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    med_d = int(np.median(depths_to_half))
    ax.axvline(med_d, color="red", lw=2, linestyle="--")
    ax.text(med_d+0.1, counts_d.max()*0.9,
            f"mediana = {med_d} kroków", color="red", fontsize=11, fontweight="bold")
else:
    ax.text(0.5, 0.5, "Żadna próbkowana funkcja\nnie osiąga 50% systemu",
            ha="center", va="center", transform=ax.transAxes, fontsize=14)
    ax.set_title("(B) Głębokość 50% — brak epicentrów w próbce", fontweight="bold")

fig.suptitle("H2. Szybkość propagacji — w ilu krokach zmiana zalewa system?",
             fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
save(fig, "08_h2_propagation_speed")
plt.show()


## H3. Blast-radius heatmapa — które pary subsystemów są sprzężone?

Dla każdego subsystemu *źródłowego* pytamy: *"gdy zmienimy losową funkcję w tym subsystemie,
jaki % funkcji w subsystemie docelowym zostanie dotkniętych?"*

To ujawnia **ukryte architekturalne sprzężenia** — takie które nie są widoczne z samego
grafu zależności (bo działają przez pośredników).


In [ ]:
# Pre-compute subsystem dla każdego node-indeksu
sub_arr = metrics["subsystem"].fillna("unknown").values

# Top subsystemy (po liczbie węzłów) — bierzemy top 14 dla czytelności
top_subs = (metrics["subsystem"].value_counts().head(14).index.tolist())
sub_to_nodes = defaultdict(list)
for idx, sub in enumerate(sub_arr):
    sub_to_nodes[sub].append(idx)

# Dla każdego subsystemu: węzły z obliczonym PC (próbka)
# Blast-radius[src_sub][tgt_sub] = mediana PC (tylko na węzłach tgt_sub)
print("Buduję blast-radius matrix…")
BR = np.zeros((len(top_subs), len(top_subs)))

for i, src_sub in enumerate(top_subs):
    src_nodes = [n for n in sub_to_nodes[src_sub]
                 if not np.isnan(metrics.iloc[n]["propagation_cost"])]
    if not src_nodes:
        continue
    # Dla każdego węzła w src_sub: policz ilu węzłów z tgt_sub dosięga
    # Używamy forward_reach który daje SET — ale dla szybkości operujemy na
    # pełnych BFS z zapamiętaniem visited
    # Próbka: max 30 węzłów z każdego subsystemu źródłowego
    rng3 = np.random.default_rng(i)
    sample_src = rng3.choice(src_nodes,
                              size=min(30, len(src_nodes)), replace=False)
    for j, tgt_sub in enumerate(top_subs):
        tgt_set = set(sub_to_nodes[tgt_sub])
        if not tgt_set:
            continue
        hit_fracs = []
        for src_v in sample_src:
            # BFS w przód — zlicz ilu z tgt_sub odwiedzono
            visited = {int(src_v)}
            frontier = [int(src_v)]
            hit = 0
            while frontier:
                nxt = []
                for u in frontier:
                    for v in out_adj[u]:
                        if v not in visited:
                            visited.add(v)
                            nxt.append(v)
                            if v in tgt_set:
                                hit += 1
                frontier = nxt
            hit_fracs.append(hit / len(tgt_set))
        BR[i, j] = np.median(hit_fracs)
    print(f"  {src_sub[:30]:<30s}  done (row {i+1}/{len(top_subs)})")

print("Blast-radius matrix gotowa.")


In [ ]:
# Skróć nazwy subsystemów do etykiet
def short_sub(s):
    # Usuń powtarzający się prefix
    parts = s.split("/")
    if len(parts) >= 2:
        return "/".join(parts[:2]) if len("/".join(parts[:2])) <= 20 else parts[-1]
    return s

short_labels = [short_sub(s) for s in top_subs]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Panel A: surowa blast-radius heatmapa ---
ax = axes[0]
mask_diag = np.eye(len(top_subs), dtype=bool)
sns.heatmap(BR, annot=True, fmt=".2f",
            xticklabels=short_labels, yticklabels=short_labels,
            cmap="rocket_r", vmin=0, vmax=1,
            linewidths=0.8, linecolor="white",
            cbar_kws={"label": "mediana frakcji tgt_sub dotknięta zmianą"},
            ax=ax, annot_kws={"size": 9})
ax.set_xlabel("Subsystem DOCELOWY (co zostaje dotknięte)")
ax.set_ylabel("Subsystem ŹRÓDŁOWY (gdzie zmiana jest wprowadzona)")
ax.set_title("(A) Blast-radius: zmiana w → wpływ na ↓", fontweight="bold")
plt.setp(ax.get_xticklabels(), rotation=40, ha="right", fontsize=9)
plt.setp(ax.get_yticklabels(), fontsize=9)

# --- Panel B: "Kto jest ofiarą"? — kolumnowe sumy (vulnerability score) ---
ax = axes[1]
# Vulnerability: średni %, ile z moich węzłów jest dotkniętych przez zmiany INNYCH
np.fill_diagonal(BR, 0)
vuln_score = BR.mean(axis=0)    # ile średnio dosięga mnie zmiana z innych subsystemów
impact_score = BR.mean(axis=1)  # ile średnio dotykam innych

sort_idx = np.argsort(-vuln_score)
y = np.arange(len(top_subs))
ax.barh(y, vuln_score[sort_idx], color=WONG[3], edgecolor="black",
        label="Vulnerability (ile zmian z zewnątrz mnie dosięga)")
ax.barh(y, -impact_score[sort_idx], color=WONG[0], edgecolor="black",
        label="Impact (ile zmian ode mnie dosięga innych)")
ax.set_yticks(y)
ax.set_yticklabels([short_labels[i] for i in sort_idx], fontsize=9)
ax.set_xlabel("Średnia frakcja osiągniętych węzłów")
ax.set_title("(B) Vulnerability vs Impact per subsystem", fontweight="bold")
ax.axvline(0, color="black", lw=1)
ax.legend(fontsize=10, loc="lower right")
ax.grid(axis="x", alpha=0.3)

fig.suptitle("H3. Blast-radius — które subsystemy zarażają i które są podatne?",
             fontsize=17, fontweight="bold", y=1.01)
plt.tight_layout()
save(fig, "08_h3_blast_radius")
plt.show()


## H4. Fragility Score — prawdziwe epicentra

Sam wysoki PC to za mało — chcemy funkcje które są jednocześnie:
1. **Duży stożek wpływu** (wysoki PC)
2. **Leżą na wielu ścieżkach** (wysoki betweenness lub in-degree)
3. **Łączą różne subsystemy** (wysoka entropia subsystemów sąsiadów z notebook 07)

$$FS(v) = PC(v) \cdot \log(1 + \text{in\_degree}(v)) \cdot H_{\text{subsys}}(v)$$

Funkcje z najwyższym FS to **single points of failure** jądra Linuxa.


In [ ]:
# Entropia subsystemów in-sąsiadów
unique_subs, sub_int = np.unique(sub_arr, return_inverse=True)
n_subs = len(unique_subs)

print("Liczę entropię subsystemową (wydajna wersja)…")
t0 = time.time()
src_idxs_arr = edges_idx["src_idx"].values
tgt_idxs_arr = edges_idx["tgt_idx"].values
src_sub_int  = sub_int[src_idxs_arr]

caller_dist = {}
for u_int, v in zip(src_sub_int, tgt_idxs_arr):
    if v not in caller_dist:
        caller_dist[v] = {}
    d = caller_dist[v]
    d[u_int] = d.get(u_int, 0) + 1

def entropy_dict(d):
    counts = np.array(list(d.values()), dtype=np.float64)
    total  = counts.sum()
    if total == 0: return 0.0
    p = counts / total
    return float(-np.sum(p * np.log2(p + 1e-12)))

H_sub = np.zeros(N)
n_callers_arr = np.zeros(N, dtype=int)
for v_idx, d in caller_dist.items():
    H_sub[v_idx] = entropy_dict(d)
    n_callers_arr[v_idx] = sum(d.values())
del caller_dist

metrics["H_subsys"] = H_sub
metrics["n_callers"] = n_callers_arr
print(f"Entropia gotowa ({time.time()-t0:.0f}s)")

# Fragility Score
pc_vals = metrics["propagation_cost"].fillna(0).values
in_deg  = metrics["in_degree"].values
h_vals  = metrics["H_subsys"].values

FS = pc_vals * np.log1p(in_deg) * h_vals
metrics["fragility_score"] = FS

# Top 20
top_fs = metrics.nlargest(20, "fragility_score")[
    ["func_name","subsystem","in_degree","out_degree",
     "propagation_cost","H_subsys","fragility_score"]
]
print("\n=== TOP 20 EPICENTRÓW (Fragility Score) ===")
print(top_fs.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 8))

# --- Panel A: top 20 epicentrów ---
ax = axes[0]
top20 = metrics.nlargest(20, "fragility_score").iloc[::-1]
colors_bar = plt.cm.get_cmap("rocket_r")(
    np.linspace(0.2, 0.9, len(top20)))

bars = ax.barh(range(len(top20)),
               top20["fragility_score"],
               color=colors_bar, edgecolor="black", linewidth=1)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(
    [f"{r.func_name[:28]}" for _, r in top20.iterrows()],
    fontsize=10)
ax.set_xlabel("Fragility Score  =  PC × log(1+in_deg) × H_subsys")
ax.set_title("(A) TOP 20 — Single Points of Failure jądra Linuxa",
             fontweight="bold")
ax.grid(axis="x", alpha=0.3)

# Mini-lollipop: zaznacz PC, log(deg), H
for i, (_, row) in enumerate(top20.iterrows()):
    # Małe kółko: rozmiar = H_subsys
    ax.scatter(row["fragility_score"] * 1.02, i,
               s=80 * row["H_subsys"] + 10,
               c=[WONG[2]], edgecolors="black", linewidths=0.8,
               zorder=5)

# --- Panel B: trójwymiarowy scatter FS ---
ax = axes[1]
has_pc = metrics["propagation_cost"].notna() & (metrics["n_callers"] >= 5)
sub_df = metrics[has_pc].sample(min(8000, has_pc.sum()), random_state=1)

sc = ax.scatter(sub_df["in_degree"] + 1,
                sub_df["propagation_cost"],
                s=20 + 60 * sub_df["H_subsys"],
                c=sub_df["fragility_score"],
                cmap="rocket_r", alpha=0.6,
                edgecolors="none", rasterized=True)
ax.set_xscale("log")
ax.set_xlabel("in-degree + 1  (log)")
ax.set_ylabel("Propagation Cost PC(v)")
ax.set_title("(B) FS w przestrzeni (in-deg, PC) — rozmiar bąbla = H_subsys", fontweight="bold")
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Fragility Score")
ax.grid(alpha=0.3)

# Zaznacz top 5
top5 = metrics.nlargest(5, "fragility_score")
for _, row in top5.iterrows():
    if np.isnan(row["propagation_cost"]): continue
    ax.scatter(row["in_degree"]+1, row["propagation_cost"],
               s=300, marker="*", c="gold",
               edgecolors="black", linewidths=1.5, zorder=10)
    ax.annotate(row["func_name"][:18],
                (row["in_degree"]+1, row["propagation_cost"]),
                fontsize=9, fontweight="bold",
                xytext=(8, 4), textcoords="offset points",
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          ec="black", alpha=0.85))

fig.suptitle("H4. Fragility Score — prawdziwe epicentra zmian w jądrze Linuxa",
             fontsize=17, fontweight="bold", y=1.01)
plt.tight_layout()
save(fig, "08_h4_fragility_score")
plt.show()


## HERO figure — "The 0.01% that rules them all"

Jedna wielka figura pod poster: pokazuje wyraźnie TEZĘ.
Narysujemy: top epicentra na tle całego rozkładu PC, z annotacją jaka frakcja systemu
jest zagrożona przez każde z nich.


In [ ]:
fig = plt.figure(figsize=(18, 10))

# Układ: 60% szerokości dla głównego panelu, 40% dla listy epicentrów
gs = fig.add_gridspec(2, 3, width_ratios=[2, 0.05, 1.2],
                      hspace=0.45, wspace=0.35)
ax_main  = fig.add_subplot(gs[:, 0])
ax_cbar  = fig.add_subplot(gs[:, 1])
ax_top   = fig.add_subplot(gs[0, 2])
ax_layer = fig.add_subplot(gs[1, 2])

# ── Główny panel: scatter in-deg × PC, kolor=FS ──────────────────────────
has_pc = metrics["propagation_cost"].notna() & (metrics["in_degree"] >= 0)
sub_df = metrics[has_pc].sample(min(15_000, has_pc.sum()), random_state=9)

sc = ax_main.scatter(
    sub_df["in_degree"] + 1,
    sub_df["propagation_cost"],
    s=8, c=sub_df["fragility_score"],
    cmap="rocket_r", alpha=0.55,
    edgecolors="none", rasterized=True,
    norm=mcolors.LogNorm(
        vmin=max(sub_df["fragility_score"].clip(lower=1e-9).min(), 1e-9),
        vmax=sub_df["fragility_score"].max()))

# Colorbar w dedykowanym axes
cb = plt.colorbar(sc, cax=ax_cbar)
cb.set_label("Fragility Score (log)", fontsize=12)

# Zaznacz top 10 epicentrów
top10 = metrics.nlargest(10, "fragility_score")
star_colors = plt.cm.get_cmap("YlOrRd")(np.linspace(0.5, 0.95, len(top10)))
for rank, (_, row) in enumerate(top10.iterrows()):
    if np.isnan(row["propagation_cost"]): continue
    ax_main.scatter(
        row["in_degree"]+1, row["propagation_cost"],
        s=500, marker="*",
        c=[star_colors[rank]], edgecolors="black",
        linewidths=1.5, zorder=10)
    if rank < 5:
        ax_main.annotate(
            f"#{rank+1} {row['func_name'][:20]}",
            (row["in_degree"]+1, row["propagation_cost"]),
            fontsize=9.5, fontweight="bold",
            xytext=(12, 4), textcoords="offset points",
            arrowprops=dict(arrowstyle="-", lw=1, color="#555"),
            bbox=dict(boxstyle="round,pad=0.25", fc="white",
                      ec="black", alpha=0.92, lw=0.8))

# Linie progowe
for pc_thresh, lbl, col in [(0.5,"50% systemu","#c0392b"),
                             (0.1,"10% systemu","#e67e22")]:
    n_nodes = (metrics["propagation_cost"].fillna(0) >= pc_thresh).sum()
    ax_main.axhline(pc_thresh, color=col, lw=1.8, linestyle="--", alpha=0.7)
    ax_main.text(1.1, pc_thresh+0.01,
                 f"{n_nodes:,} funkcji → ≥{int(pc_thresh*100)}%",
                 color=col, fontsize=10, fontweight="bold")

ax_main.set_xscale("log")
ax_main.set_xlabel("in-degree (skala log)", fontsize=14, fontweight="bold")
ax_main.set_ylabel("Propagation Cost PC(v)", fontsize=14, fontweight="bold")
ax_main.set_title(
    "The 0.01% that rules them all — Które funkcje jądra Linuxa rządzą propagacją zmian?",
    fontsize=16, fontweight="bold", pad=15)
ax_main.grid(alpha=0.25, which="both")

# ── Top-right: rankinig epicentrów ────────────────────────────────────────
top8 = metrics.nlargest(8, "fragility_score").reset_index(drop=True)
ax_top.axis("off")
table_data = []
for i, row in top8.iterrows():
    sub_short = "/".join(str(row["subsystem"]).split("/")[:2])[:18]
    table_data.append([
        f"#{i+1}",
        row["func_name"][:22],
        sub_short,
        f"{row['propagation_cost']:.3f}" if not np.isnan(row["propagation_cost"]) else "—",
        f"{row['fragility_score']:.2f}",
    ])
tbl = ax_top.table(
    cellText=table_data,
    colLabels=["#","Funkcja","Subsystem","PC","FS"],
    cellLoc="left", loc="center",
    bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#0072B2")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EFF6FF")
    cell.set_edgecolor("white")
ax_top.set_title("Top 8 epicentrów", fontsize=12,
                  fontweight="bold", pad=4)

# ── Bottom-right: warstwy propagacji dla top epicentrum ──────────────────
top1_idx = int(metrics["fragility_score"].idxmax())
layers1 = bfs_layers(top1_idx, out_adj, max_depth=20)
cumsum1  = np.cumsum(layers1) / N * 100
fname1   = metrics.iloc[top1_idx]["func_name"]

ax_layer.fill_between(range(1, len(cumsum1)+1), cumsum1,
                       alpha=0.35, color=WONG[6])
ax_layer.plot(range(1, len(cumsum1)+1), cumsum1,
              color=WONG[6], lw=3, marker="o", markersize=7)
ax_layer.axhline(50, color="red", lw=1.5, linestyle=":")
ax_layer.set_xlabel("Głębokość propagacji (kroki)")
ax_layer.set_ylabel("% systemu")
ax_layer.set_title(f"Propagacja #1: {fname1[:26]}", fontsize=11, fontweight="bold")
ax_layer.grid(alpha=0.3)
ax_layer.set_ylim(0, 105)

plt.suptitle(
    "Analiza sieci wywołań jądra Linux — Epicentra zmian",
    fontsize=20, fontweight="bold", y=1.01)

save(fig, "08_HERO_epicenters")
plt.show()


## Wnioski — co powiedzieć na posterze?

### Teza potwierdzona (lub obalona) przez dane:

> *"Zmiana <0.01% funkcji jądra może dosięgnąć >50% systemu"*

Sprawdź w swoich danych:
- Ile funkcji ma PC > 0.5? Ile to % z N=466k?
- Jakie funkcje są w top 10 FS? Czy to intuicyjne? (`mutex_lock`? `kmalloc`? `schedule`?)

### Co to znaczy architekturalnie?

| Obserwacja | Interpretacja |
|------------|---------------|
| PC ma gruby ogon (heavy tail) | Kilka funkcji ma nieproporjonalnie duży zasięg |
| Epicentra leżą w `kernel/*` i `mm/*` | Niskopoziomowe prymitywy są "węzłami synchronizacji" |
| Blast-radius `kernel→drivers` >> `drivers→kernel` | Architektura jest skierowana — naruszenia warstw są rzadkie |
| Top FS ≠ Top in-degree | Sam stopień nie wystarczy — liczy się *gdzie* funkcja leży |

### Jedna zdanie na plakat:
> *"W jądrze Linuxa istnieje ~X funkcji (Y% systemu), których modyfikacja może propagować zmiany do ponad połowy bazy kodu — nazywamy je 'epicentrami'."*

### Dalsze kierunki (jeśli jest czas):
- Porównaj FS z historią commitów Linuxa — czy epicentra są częściej modyfikowane?
- Czy funkcje z najwyższym FS są lepiej udokumentowane / mają więcej testów?
- Porównaj propagation cost z wersją jądra 4.x vs 6.x — czy rośnie?
